In [9]:
import pandas as pd
from fpdf import FPDF
import os
import glob
from pathlib import Path
import requests
from io import BytesIO

CRIAR DATA FRAME:

INSERIR CAMINHO DA PLANILHA ABAIXO COM A SEGUINTE ESTRUTURA: CAMINHO DO ARQUIVO, NOME DA PLANILHA

PARA GERAR MAIS DE UM MATERIAL BASTA CRIAR MAIS DE UM DATA_FRAME

In [ ]:
nome_planilha = "SIMILARES_LINHA"

df_quadradinho = pd.read_excel(
    r"C:\Users\Victor.figueiredo\Downloads\PRODUTOS.xlsx",
    sheet_name= nome_planilha,
    dtype={'PRODUTO': str, 'COR_PRODUTO': str}
)

# Ordena de forma estável: pages saem em ordem de LINHA da âncora e,
# dentro de cada âncora, os 5 similares respeitam o RANK (1..5).
df_quadradinho = df_quadradinho.sort_values(by=['LINHA', 'ID', 'RANK'], ascending=True)

MAPEAMENTO FOTOS: ALTERAR O CAMINHO PARA MAPEAR AS FOTOS DA COLEÇÃO DESEJADA

In [ ]:
# 1. Defina o caminho base (o ** e recursive=True buscam em todas as subpastas)

caminho_col1 = r'\**\*.jpg'


mapa_colecao1 = {os.path.basename(f): f for f in glob.glob(caminho_col1, recursive=True)}





mapa_fotos = mapa_colecao1 



print(f"Mapeamento concluído! {len(mapa_fotos)} fotos encontradas.")

def obter_imagem(url_foto):
    try:
        # Faz a requisição para a URL
        response = requests.get(url_foto, timeout=60)
        # Transforma o conteúdo binário em um "arquivo de memória"
        image_data = BytesIO(response.content)
        return image_data
    except Exception as e:
        print(f"Erro ao baixar a imagem: {e}")
        return None

Mapeamento concluído! 7345 fotos encontradas.


In [ ]:
def gerar_catalogo(df, nome_aba):
    # criando página vazia
    pdf = FPDF(orientation='L', unit='mm', format='A4')
    pdf.add_page()

    # parâmetros de layout (mesmos do formato 1 + 5)
    largura_pagina = 297  # A4 paisagem
    X_inicial = 10
    Y_inicial = 15
    largura = 35
    espacamento = 90              # vertical entre âncora e linha de similares
    espacamento_horizontal = 60   # horizontal entre similares
    X_centralizado = (largura_pagina - largura) / 2

    primeira_pagina = True

    # 1 página por âncora (groupby preserva a ordem do sort_values aplicado no df)
    for id_ancora, grupo in df.groupby('ID', sort=False):
        if not primeira_pagina:
            pdf.add_page()
        primeira_pagina = False

        grupo = grupo.sort_values('RANK', ascending=True).head(5)
        ancora = grupo.iloc[0]

        # Monta os 6 quadros da página: 1 âncora centralizada + 5 similares.
        # Âncora: ID em negrito sob a foto, com DESC_PRODUTO e CLUSTER_PRODUTO.
        # Foto da âncora vem da URL somalabs via obter_imagem.
        # Similares: ID_SIMILAR em negrito, com DESC_PRODUTO_SIMILAR,
        # CLUSTER_PRODUTO_SIMILAR e SIMILARIDADE; foto vem do mapa local.
        quadros = []
        quadros.append({
            'X': X_centralizado,
            'Y': Y_inicial,
            'linha_badge': ancora['LINHA'],
            'produto': ancora['PRODUTO'],
            'cor': ancora['COR_PRODUTO'],
            'url_foto': f"PATH_PICS/{ancora['PRODUTO']}/{ancora['COR_PRODUTO']}",
            'titulo': ancora['ID'],
            'descricoes': [
                str(ancora['DESC_PRODUTO']),
                str(ancora['CLUSTER_PRODUTO']),
                str(ancora['COLECAO']),
            ],
        })
        for pos, (_, par) in enumerate(grupo.iterrows()):
            quadros.append({
                'X': X_inicial + pos * espacamento_horizontal,
                'Y': Y_inicial + espacamento,
                'linha_badge': par['LINHA_SIMILAR'],
                'produto': par['PRODUTO_SIMILAR'],
                'cor': par['COR_PRODUTO_SIMILAR'],
                'colecao' : par['COLECAO_SIMILAR'],
                'url_foto': None,
                'titulo': par['ID_SIMILAR'],
                'descricoes': [
                    str(par['DESC_PRODUTO_SIMILAR']),
                    str(par['CLUSTER_PRODUTO_SIMILAR']),
                    str(par['COLECAO_SIMILAR']),
                    f"Similaridade: {par['SIMILARIDADE']:.1%}",
                ],
            })

        # Renderiza cada quadro (badge LINHA + foto + título + descrições)
        for q in quadros:
            X_foto, Y_foto = q['X'], q['Y']

            # BADGE LINHA (acima da foto, fundo bege)
            pdf.set_fill_color(246, 234, 214)
            pdf.set_font('Helvetica', 'B', 6)
            pdf.set_xy(X_foto, Y_foto - 5)
            pdf.cell(w=largura, h=4, text=str(q['linha_badge']),
                     new_x="RIGHT", new_y="TOP", align='C', fill=True)

            # FOTO
            url_foto = q.get('url_foto')
            if url_foto:
                # Âncora: download via obter_imagem (retorna BytesIO ou None)
                caminho_foto_prod = obter_imagem(url_foto)
            else:
                # Similares: lookup no mapa de fotos local da coleção
                prod_fmt = str(q['produto']).strip()
                cor_fmt = str(q['cor']).strip()
                nome_esperado = f"{prod_fmt}-{cor_fmt}.jpg"
                nome_esperado_2 = f"{prod_fmt}.{cor_fmt}.jpg"
                nome_final = nome_esperado if nome_esperado in mapa_fotos else nome_esperado_2
                caminho_foto_prod = mapa_fotos.get(nome_final)

            if caminho_foto_prod is not None:
                try:
                    pdf.image(caminho_foto_prod, x=X_foto, y=Y_foto, w=largura)
                except Exception:
                    print(f"Erro ao renderizar imagem de {q['titulo']}")
                    pdf.rect(X_foto, Y_foto, largura, largura + 20)
                    pdf.set_y(Y_foto + largura - 11)
                    pdf.set_x(X_foto)
                    pdf.set_font('Helvetica', '', 7)
                    pdf.cell(w=largura, h=5, text="Imagem Não Encontrada",
                             new_x="RIGHT", new_y="TOP", align='C')
            else:
                print(f"Imagem de {q['titulo']} não encontrada ")
                pdf.rect(X_foto, Y_foto, largura, largura + 20)
                pdf.set_y(Y_foto + largura - 11)
                pdf.set_x(X_foto)
                pdf.set_font('Helvetica', '', 7)
                pdf.cell(w=largura, h=5, text="Imagem Não Encontrada",
                         new_x="RIGHT", new_y="TOP", align='C')

            # TÍTULO (negrito, primeira linha de texto sob a foto = ID / ID_SIMILAR)
            pdf.set_y(Y_foto + largura + 20)
            pdf.set_x(X_foto)
            pdf.set_font('Helvetica', 'B', 7)
            pdf.cell(w=largura, h=5, text=str(q['titulo']),
                     new_x="RIGHT", new_y="TOP", align='C')

            # DEMAIS LINHAS DE TEXTO
            pdf.set_font('Helvetica', '', 6)
            y_offset = 25
            for texto in q['descricoes']:
                pdf.set_y(Y_foto + largura + y_offset)
                pdf.set_x(X_foto)
                pdf.cell(w=largura, h=3, text=str(texto),
                         new_x="RIGHT", new_y="TOP", align='C')
                y_offset += 3

    caminho = Path(r"C:\Users\Relatorios")
    nome_arquivo = f"{nome_aba}.pdf"
    caminho_arquivo = caminho / nome_arquivo
    pdf.output(str(caminho_arquivo))

AO GERAR O CATÁLOGO INSERIR: DATA_FRAMA, NOME DA PLANILHA

In [ ]:
gerar_catalogo(df_quadradinho,nome_planilha)
#gerar_catalogo(df_quadradinho2,"PLANILHA_2")
#gerar_catalogo(df_quadradinho3,"PLANILHA_3")